# Clustering of TOP 5 European Leagues Players
**Authors:** Cezary Kuźmowicz, Filip Żebrowski, Dariusz Doktorski

## “The ball doesn’t go in by chance” ~ Johan Cruyff

Football is the most popular sport on Earth. Millions of people around the globe play it on daily basis. Most of countries have their own national leagues. But the best of the best take place in Europe. In football's nomenclature, while sharing some graphs and statistics, often used concept is "TOP 5 European Leagues". That means clearly five best national competitions in Europe. 

This group consist of English Premier League, Spanish LaLiga, Italian Serie A, German Bundesliga and French Ligue 1. In this report we will conduct clustering analysis using players statistics from season 2021/22.

In order to achieve satisfactory results many tests and visualizations will be presented. As the clustering method we've chosen k-means, mainly due to medium size of dataset and simply interpretation.

[dataset used for analysis](https://www.kaggle.com/datasets/vivovinco/20212022-football-player-stats)

### Installing necessary packages

In the beginning we have to install and access needed packages for whole analysis:

In [ ]:
from IPython.display import display

from football_clustering import (
    AnalysisConfig,
    add_cluster_labels,
    add_pca_components,
    cluster_feature_means,
    configure_notebook,
    evaluate_clusters,
    filter_by_minutes,
    fit_kmeans,
    hopkins_statistic,
    load_player_stats,
    plot_cluster_evaluation,
    plot_correlation_matrix,
    plot_distributions,
    plot_pca_clusters,
    plot_silhouette_profile,
    prepare_visualization_data,
    scale_features,
    select_features,
    validate_no_missing_values,
)

### Setting up parameters
By centralizing all our configuration constants, we eliminate "magic numbers" from our code and make it fully reproducible and easy to tweak in one place.

In [ ]:
FILE_PATH = "data/2021-2022 Football Player Stats.csv"

config = AnalysisConfig(file_path=FILE_PATH)

SELECTED_FEATURES = list(config.selected_features)
VISUALIZATION_FEATURES = list(config.visualization_features)

configure_notebook(config.random_seed)

# CLUSTERING PREPARATIONS

## Data Preparation

### Loading dataset into Python

The dataset comes from Kaggle. It was prepared based on data from [fbref](https://fbref.com/en/) - online website gathering huge amount of informations from plenty of sports.

In [ ]:
# Load the dataset using shared CSV formatting parameters.
raw_stats = load_player_stats(config)

The raw dataset includes nearly 3000 observations (individual players), each described by 143 variables. It's important to add that every statistic is calculated "per 90 minutes". This means that the dataset's author already unified the data.

The first step during our data preparation will be excluding footballers who played less than a specific threshold of minutes through the whole season. That operation will assure us that we don't analyze a player who, for example, scored 4 goals in his only played match.

In [ ]:
# Filter players based on the configured minimum minutes played.
stats_filtered = filter_by_minutes(raw_stats, config.min_minutes_played)

By conducting such an operation we got rid of nearly 600 observations. That will definitely improve overview of our dataset and quality of analysis.

Before removing some of them, let's dive into overview of the data

In [ ]:
display(stats_filtered.head())
display(stats_filtered.describe())
print(stats_filtered.info())

As you can see, there is too much information. Analyzing dataset with such a number of variables is very impractical. But there are some overall insights are can obtain:
   
* many variables are right-skewed
* in almost every statistic max value is hard outlier. It isn't error; that's just world-class players

In conducting clustering important part is assuring that none NAs occur

In [ ]:
# Print variables with missing values (if any) and fail before clustering if any exist.
missing_vals = validate_no_missing_values(stats_filtered)
print(missing_vals)
print("Data validation passed: No missing values found.")

As we can see - there are no NA values. Probably the author from Kaggle already handled it.

## Handling number of variables

In our dataset we have nearly 150 columns. That is pretty impressive amount but plenty of them are too detailed/provide no valuable information. In order to choose the best we studied every columns. What's more, we're football fans for more than 12 years. Thanks to that experience we were able to choose the most valuable variables. 

After plenty of testing (including calculating Hopkins statistics for each set of columns), we decided to limit number of variables from 143 to 8. Part of Data Scientist's job is using previous experience and knowledge to better understand and solve problems. So did we here!

We've tried to choose as diverse and valuable variables as we could. Let's decode them:  
* **Shots**: shots taken by player, per 90 min  
* **PasTotAtt**: total number of pass attempts, per 90 min   
* **Assists**: assists given by player, per 90 min   
* **Tkl**: number of times a ball has been picked up (tackled), per 90 min   
* **Block**: number of times player has blocked a pass/shot, per 90 min    
* **Fls**: number of fauls comitted by player, per 90 min    
* **Off**: number of times player has been caught on offside position, per 90 min    
* **AerWon%**: share of won aerial head duels, in %

By creating new data frame with less variables, we could easily continue the analysis.

In [ ]:
reduced_stats = select_features(stats_filtered, SELECTED_FEATURES)
reduced_stats.head()

### Hopkins statistics
In order to check how clusterable is the dataset, we'll compute Hopkins test. It compares the distances between randomly sampled points in the dataset and synthetic points generated uniformly in the feature space. A value close to **1** indicates strong clustering tendency, while a value near **0.5** suggests the data is uniformly distributed and unlikely to form meaningful clusters.

In [ ]:
hopkins_val = hopkins_statistic(
    reduced_stats,
    sample_ratio=config.hopkins_sample_ratio,
    n_neighbors=config.hopkins_n_neighbors,
)
print(f"Hopkins Statistic: {hopkins_val:.4f}")

The value of Hopkins statistic is high. That's good predictor for our clustering.

## Data Visualisation

In order present some of the statistics of charts, we'll create data frame variable for those purposes.

In [ ]:
player_stats = prepare_visualization_data(stats_filtered, VISUALIZATION_FEATURES)

### Visualizing Player Statistics
To understand our dataset better and ensure results aren't biased by outliers, we will visualize a few key statistics:

* **Minutes played**: It's important to see if our results won't be biased by footballers who played very limited time.
* **Goals**: We multiply "goals/90min" by "played 90s" to present the most goalscoring players.
* **Passes**: How many passes players make per 90 minutes.
* **Aerial Duels Won (%)**: A key attribute for center-backs and target-man forwards. 

In [ ]:
plot_distributions(player_stats)

### Interpretation of the Distributions

* **Minutes Played**: The played time is more or less equally distributed across the spectrum. The biggest differences and spikes are visible at the very beginning (players with very few minutes) and the end (undisputed starters who play almost every match), which is highly intuitive.
* **Goals (Total)**: As expected, the vast majority of players (defenders, goalkeepers, defensive midfielders) didn't score any goals or scored very few. The plot features an extremely long right tail—these aren't errors in the dataset, but rather the super-star forwards who score 20+ goals a season.
* **Passes (per 90 min)**: This statistic is much more normally distributed than goals. Passing is a core mechanic for almost all players on the pitch, not just a specific trait of midfielders. We can observe a slight right skew, which represents deep-lying playmakers and center-backs who circulate the ball the most frequently.
* **Aerial Duels Won (%)**: This displays a somewhat normal distribution centered around the 40-60% mark. However, there are notable and massive spikes at the absolute extreme 0%. This is mostly due to players who attempted very few aerial duels throughout the entire season, which heavily distorted their personal win percentages.

### Correlation Matrix
Before moving to clustering, let's verify if there is strong collinearity among the chosen features.

In [ ]:
plot_correlation_matrix(reduced_stats, "Correlation Matrix of Selected Features")

### Interpretation of the Correlation Matrix

Analyzing the correlation matrix gives us confidence that our selected variables are well-suited for clustering:

* **Logical Groupings**: We can clearly see moderate to strong positive correlations between offensive metrics (e.g., `Shots`, `Assists`, and `Off`), as well as between defensive metrics (e.g., `Tkl` and `Blocks`). This makes sense, as players usually specialize in either attack or defense.
* **Negative Correlations**: There are noticeable negative correlations between offensive and defensive stats (e.g., `Shots` vs. `Blocks`). This divergence is exactly what K-Means needs to effectively separate attackers from defenders.
* **No Extreme Multicollinearity**: Crucially, there are no correlations extremely close to 1.0 or -1.0. If two variables were perfectly correlated, one of them would be redundant and could artificially over-weight a specific trait during the K-Means distance calculations. The current balance is ideal for creating distinct, meaningful clusters.

# CLUSTERING ANALYSIS

## Pre-clustering details
### Scaling the data

Scaling data is necessary - variable such as "AerWon%" would have stronger impact on results of analysis due to it's format (range from 0 to 100). Rest variables are calculated per 90 minutes.

In [ ]:
stats_scaled, scaler = scale_features(reduced_stats, SELECTED_FEATURES)

display(stats_scaled.head())
display(stats_scaled.describe().round(2))

After scaling we could see that our data is more reliable showing variance in variables. It'll be perfect for clustering.

### Optimal number of clusters

In order to get optimal number of clusters, we will use the Elbow Method (Variance Explained) and Silhouette Scores. Decision about number of clusters is not easy, but checking multiple metrics provides a clearer picture.

In [ ]:
cluster_evaluation = evaluate_clusters(
    stats_scaled,
    max_k=config.max_clusters,
    random_seed=config.random_seed,
    n_init=config.kmeans_n_init,
)

display(cluster_evaluation.round(4))
plot_cluster_evaluation(cluster_evaluation)

Based on both the Elbow Method and Silhouette Scores, 3 clusters seem to be the most optimal choice. The WCSS curve starts flattening out after k=3, and the Silhouette Score is also highest for k=3.

## K-Means Clustering

In [ ]:
kmeans_final = fit_kmeans(
    stats_scaled,
    n_clusters=config.optimal_k,
    random_seed=config.random_seed,
    n_init=config.kmeans_n_init,
)

reduced_stats = add_cluster_labels(reduced_stats, kmeans_final.labels_)

### Silhouette Profile
To check how well an observation fits into its assigned cluster, we will calculate and visualize the Silhouette width. It has a range from -1 to 1, and we can interpret certain values in this way:   

* **s(i) ~ +1**: observation is much closer to points in its own cluster than to points in other clusters (good fit, perfect situation).   
* **s(i) ~ 0**: observation is on the decision boundary between two neighboring clusters.   
* **s(i) ~ -1**: observation is assigned to the wrong cluster.   

Knowing the theory behind Silhouette width, we can now calculate and visualize it.

In [ ]:
avg_silhouette = plot_silhouette_profile(
    stats_scaled,
    kmeans_final.labels_,
    config.optimal_k,
)

Average Silhouette Width: 0.34

**Silhouette Plot Interpretation:**
The dashed red line represents the average silhouette score for the entire dataset. We can observe that all three clusters have a majority of observations passing this average line, which is a sign of good structural integrity. 

* **Cluster 2** (the thickest shape at the top) has the largest number of observations. It maintains a solid positive shape, though its maximum silhouette values are slightly lower than the other clusters.
* **Cluster 1** (the thinnest shape in the middle) has the fewest observations but shows the highest cohesion. Its values stretch the furthest to the right (reaching nearly 0.45), indicating these data points are extremely distinct and well-matched to their cluster.
* **Cluster 0** (the medium-thick shape at the bottom) shows a healthy structure with many points stretching well past the average line. There are very few negative values across the board, meaning almost no players were outright assigned to the wrong cluster.

### PCA Visualization
Since our dataset has 8 dimensions, we need to reduce it to 2D for visualization purposes using Principal Component Analysis (PCA).

In [ ]:
reduced_stats = add_pca_components(reduced_stats, stats_scaled)
plot_pca_clusters(reduced_stats)

### Cluster Interpretation
To understand what each cluster represents, let's look at the mean values of the original features for each cluster.

In [ ]:
cluster_means = cluster_feature_means(reduced_stats, SELECTED_FEATURES)
display(cluster_means.round(2))

### Cluster Interpretation

**Cluster 0 - Offensive Players:** Second largest cluster. As its name states, it's characterized by huge positive variation from standard metrics describing scoring/creating actions. Shots, assists, and offsides are the highest here. Footballers in this cluster are the main focus of fans and press - they are the ones delivering magic to the matches. We have to notice that they rarely help in defense (fewest tackles and blocks).

**Cluster 1 - Goalkeepers:** The smallest cluster with the strongest fit. Because there is only one goalkeeper per team on the pitch, this is naturally the smallest group. Their standard metrics are highly distinct from outfield players, which explains why they have the highest silhouette score (strongest fit). They have zero or near-zero offensive and defensive outfield stats, but they heavily inflate passing metrics from goal kicks and distribution from the back.

**Cluster 2 - Defensive Players:** The largest cluster with moderate fit. We've named it defensive because most statistics have reversed values compared to the offensive players. It contains a large group of players - not only pure defenders but also plenty of defensive-minded midfielders. They tend to shoot less and pass more (while having fewer assists). Clear defensive statistics like blocks and tackles (Tkl) are also highest in this cluster. Those players are also winning most aerial (head) duels; that could be influenced by the typical height of center-backs.

## Conclusions 

In this report, we have gone through every process of clustering analysis. Started with loading data and choosing the most appropriate variables. Then some visualizations to better understand the whole dataset. In the end, the crème de la crème, proper clustering.
   
The results could be, for someone interested in football, pretty intuitive. This analysis also confirms that expertise/domain knowledge strongly agrees with independent, unsupervised machine learning algorithms.